In [22]:
# Create Spark session and load data
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when

spark = SparkSession.builder.getOrCreate()
df = spark.read.csv("C:/Users/tf/Desktop/airbnb_data_cleaned.csv", header=True, inferSchema=True)


In [23]:
# Cleaning the price column
df = df.withColumn("price", when(col("price").rlike("^-?[0-9]*\.?[0-9]+$"), col("price")).otherwise(None))
df = df.withColumn("price", col("price").cast("double"))

# Clean review_rate_number column
df = df.withColumn("review_rate_number", when(col("review_rate_number").rlike("^-?[0-9]*\.?[0-9]+$"), col("review_rate_number")).otherwise(None))
df = df.withColumn("review_rate_number", col("review_rate_number").cast("double"))

# Clean number_of_reviews column- keep only numeric values
df = df.withColumn("number_of_reviews", when(col("number_of_reviews").rlike("^-?[0-9]*\.?[0-9]+$"), col("number_of_reviews")).otherwise(None))
df = df.withColumn("number_of_reviews", col("number_of_reviews").cast("double"))


df = df.withColumn("availability_365", when(col("availability_365").rlike("^-?[0-9]*\.?[0-9]+$"), col("availability_365")).otherwise(None))
df = df.withColumn("availability_365", col("availability_365").cast("double"))
# Remove rows where price is null 
df = df.filter(col("price").isNotNull())

df.createOrReplaceTempView("airbnb")

print("Data loaded:", df.count())


Data loaded: 99542


In [24]:
# Most expensive neighborhoods by average price
query1 = """
SELECT 
    neighbourhood_group,
    COUNT(*) as total_units,
    ROUND(AVG(price), 2) as avg_price
FROM airbnb
GROUP BY neighbourhood_group
ORDER BY avg_price DESC
"""
spark.sql(query1).show()

+-------------------+-----------+---------+
|neighbourhood_group|total_units|avg_price|
+-------------------+-----------+---------+
|              Bronx|       2607|    629.5|
|             Queens|      12802|   629.32|
|           Brooklyn|      40527|   626.19|
|          Manhattan|      42364|   623.01|
|      Staten Island|        921|   622.58|
|            brookln|          1|    580.0|
|               SoHo|          1|    233.0|
|           Rosedale|          1|    227.0|
|           Red Hook|          2|    222.0|
|            Fordham|          1|    217.0|
|            Astoria|          4|   201.75|
|               NoHo|          2|    196.5|
|        Little Neck|          2|    196.0|
|     Queens Village|          2|    193.0|
|      Cypress Hills|          2|    188.0|
|  Greenwich Village|          2|    183.0|
|          Ridgewood|          1|    183.0|
|         Greenpoint|          1|    180.0|
|      Crown Heights|          8|   174.88|
|             Inwood|          4

In [25]:
# Average price by room type
query2 = """
SELECT 
    room_type,
    COUNT(*) as total_units,
    ROUND(AVG(price), 2) as avg_price
FROM airbnb
WHERE room_type IN ('Entire home/apt', 'Private room', 'Shared room', 'Hotel room')
GROUP BY room_type
ORDER BY avg_price DESC
"""
spark.sql(query2).show()

+---------------+-----------+---------+
|      room_type|total_units|avg_price|
+---------------+-----------+---------+
|     Hotel room|        111|   663.39|
|    Shared room|       2146|   631.62|
|Entire home/apt|      52049|   625.19|
|   Private room|      44916|   625.01|
+---------------+-----------+---------+



In [26]:
query3 = """
SELECT 
    host_name,
    COUNT(*) as total_units,
    SUM(number_of_reviews) as total_reviews,
    ROUND(AVG(review_rate_number), 2) as avg_rating
FROM airbnb
WHERE host_name IS NOT NULL AND host_name != 'Unknown'
GROUP BY host_name
ORDER BY total_reviews DESC
LIMIT 10
"""

result3 = spark.sql(query3)
result3.show(truncate=False)

+---------+-----------+-------------+----------+
|host_name|total_units|total_reviews|avg_rating|
+---------+-----------+-------------+----------+
|Michael  |856        |24115.0      |3.27      |
|David    |750        |17369.0      |3.23      |
|John     |567        |15214.0      |3.21      |
|Alex     |528        |13470.0      |3.23      |
|Eric     |286        |12000.0      |3.24      |
|Chris    |367        |11970.0      |3.21      |
|Michelle |362        |11816.0      |3.25      |
|Anna     |393        |11538.0      |3.15      |
|Kevin    |268        |11524.0      |3.21      |
|Daniel   |456        |10715.0      |3.35      |
+---------+-----------+-------------+----------+



In [27]:
# Top 10 hosts by total reviews
query3 = """
SELECT 
    CASE 
        WHEN price <= 100 THEN 'Budget'
        WHEN price <= 300 THEN 'Mid-Range'
        ELSE 'Luxury'
    END as price_category,
    COUNT(*) as total_units,
    ROUND(AVG(review_rate_number), 2) as avg_rating
FROM airbnb
WHERE review_rate_number IS NOT NULL
GROUP BY price_category
"""
spark.sql(query3).show()

+--------------+-----------+----------+
|price_category|total_units|avg_rating|
+--------------+-----------+----------+
|        Budget|       4459|      3.31|
|        Luxury|      77743|      3.28|
|     Mid-Range|      17340|      3.27|
+--------------+-----------+----------+



In [28]:

query4 = """
SELECT 
    CASE 
        WHEN review_rate_number >= 4.5 THEN 'Excellent'
        WHEN review_rate_number >= 4.0 THEN 'Good'
        WHEN review_rate_number >= 3.0 THEN 'Average'
        ELSE 'Poor'
    END as rating_category,
    COUNT(*) as total_units,
    ROUND(AVG(price), 2) as avg_price
FROM airbnb
WHERE review_rate_number IS NOT NULL
GROUP BY rating_category
ORDER BY 
    CASE rating_category
        WHEN 'Excellent' THEN 1
        WHEN 'Good' THEN 2
        WHEN 'Average' THEN 3
        ELSE 4
    END
"""
spark.sql(query4).show(truncate=40)

+---------------+-----------+---------+
|rating_category|total_units|avg_price|
+---------------+-----------+---------+
|      Excellent|      22692|   620.58|
|           Good|      22567|   625.54|
|        Average|      22757|   624.75|
|           Poor|      31526|   623.83|
+---------------+-----------+---------+



In [29]:
# Lowest rated airbnb below 75

query5 = """
SELECT 
    name,
    neighbourhood_group,
    review_rate_number as rating
FROM airbnb
WHERE review_rate_number < 75 
  AND review_rate_number IS NOT NULL
  AND name RLIKE '^[A-Za-z]'
ORDER BY rating ASC
LIMIT 10
"""
spark.sql(query5).show(truncate=40)

+----------------------------------------+-------------------+------+
|                                    name|neighbourhood_group|rating|
+----------------------------------------+-------------------+------+
|    PRIVATE SPACE  L TRAIN BEDFORD STOP!|           Brooklyn|   1.0|
|          Cozy bedroom by Yankee Stadium|              Bronx|   1.0|
|     Midtown East Sutton Area Entire Apt|          Manhattan|   1.0|
|         Private room in cozy Greenpoint|           Brooklyn|   1.0|
|       Large bedroom in Harlem available|          Manhattan|   1.0|
|Great apartment with private bathroom...|          Manhattan|   1.0|
|     Great DEAL Gramercy 1 BDROOM /2beds|          Manhattan|   1.0|
|     Greenwich Village Stylish Apartment|          Manhattan|   1.0|
|Convenient 2 BRs in 4BR Apartment in ...|           Brooklyn|   1.0|
|      Cozy 2 br in sunny Fort Greene apt|           Brooklyn|   1.0|
+----------------------------------------+-------------------+------+

